# Train a tiny LLM on SEC filings

Run this notebook **on your laptop** (in WSL or any local Jupyter). It uses your RunPod API key to spin up a 1× A100 80 GB pod, uploads SEC filing data from `data/filings-2025-2026/`, runs Karpathy's [nanochat](https://github.com/karpathy/nanochat) training pipeline on the pod, and lets you chat with the result.

Total time: ~10 min (pod boot + data upload + ~5 min training).

**Before running:**

```bash
pip install runpod requests
# WSL/Linux/Mac: ssh + scp + ssh-keygen come pre-installed.
```

Run cells top to bottom. The last cell terminates the pod — don't skip it.

## 1. Paste your RunPod API key

Generate one at https://www.runpod.io/console/user/settings (you also need at least \$10 of credit; the workshop run is \$0.30–0.50).

In [1]:
import getpass, runpod
runpod.api_key = getpass.getpass("RunPod API key: ")
me = runpod.get_user()
print(f"OK, logged in as {me.get('email', '?')}")

RunPod API key:  ········


OK, logged in as ?


## 2. Constants

Tweak only if you want to change which SEC sections to upload, or push the model bigger. Defaults: just the 12 MB `market_risk` section, depth-4 GPT (~5 M params), 400 training steps. Fits in ~10 min wall clock.

In [2]:
import pathlib

DATA_DIR    = pathlib.Path("data/filings-2025-2026").resolve()

# Which parquets to upload (smallest first). Each row is one SEC filing section.
PARQUETS = [
    "market_risk_2025_2026.parquet",   # 12 MB — uploads in seconds
    # "mda_2025_2026.parquet",         # 207 MB — uncomment for richer corpus
    # "business_2025_2026.parquet",    # 290 MB
    # "risk_factors_2025_2026.parquet",# 470 MB — most stylistically distinctive
]

POD_NAME    = "zero-to-llm"
GPU_TYPE    = "NVIDIA A100 80GB PCIe"   # fallback list tried below
GPU_FALLBACK = ["NVIDIA A100-SXM4-80GB", "NVIDIA H100 80GB HBM3", "NVIDIA H100 PCIe"]
POD_IMAGE   = "runpod/pytorch:2.4.0-py3.11-cuda12.4.1-devel-ubuntu22.04"
DISK_GB     = 60

# Training shape — these are passed to nanochat's scripts/base_train.py
DEPTH         = 4         # 4 -> ~5 M params, 6 -> ~14 M, 8 -> ~30 M
NUM_STEPS     = 400       # ~5–7 min on A100 at depth=4
VOCAB_SIZE    = 4096
MAX_SEQ_LEN   = 1024
DEV_BATCH     = 8
TOTAL_BATCH   = 32768
TOK_MAX_CHARS = 20_000_000

print(f"Will upload {len(PARQUETS)} parquet(s) totaling "
      f"{sum((DATA_DIR/p).stat().st_size for p in PARQUETS)/1e6:.1f} MB")

Will upload 1 parquet(s) totaling 12.4 MB


## 3. Make sure SSH key exists & is registered with RunPod

If you don't have one at `~/.ssh/id_ed25519`, this generates one (no passphrase) and uploads the public part to your RunPod account. RunPod injects it into the pod's `authorized_keys` automatically.

In [3]:
import subprocess, pathlib

priv = pathlib.Path.home() / ".ssh" / "id_ed25519"
pub  = priv.with_suffix(".pub")
if not priv.exists():
    print(f"Generating SSH key at {priv} ...")
    priv.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["ssh-keygen", "-t", "ed25519", "-f", str(priv), "-N", "", "-q"], check=True)

pubkey = pub.read_text().strip()
runpod.update_user_settings(pubkey=pubkey)
print(f"Registered public key with RunPod ({pubkey[:50]}...)")

Generating SSH key at /home/neilcelik/.ssh/id_ed25519 ...
Registered public key with RunPod (ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIIOKUQ0CXXiYbX...)


## 4. Spin up the GPU pod

Picks the first available GPU from the preference list. The pod's default entrypoint runs JupyterLab in the background — we ignore it and SSH in directly for everything.

In [ ]:
import time

available = {g['id'] for g in runpod.get_gpus()}
gpu_type = next((g for g in [GPU_TYPE] + GPU_FALLBACK if g in available), None)
if gpu_type is None:
    raise RuntimeError(f"None of the preferred GPU types are available. Available: {sorted(available)}")
print(f"Using GPU: {gpu_type}")

pod = runpod.create_pod(
    name=POD_NAME,
    image_name=POD_IMAGE,
    gpu_type_id=gpu_type,
    gpu_count=1,
    cloud_type='ALL',
    container_disk_in_gb=DISK_GB,
    ports='22/tcp,8888/http',     # 22 for our SSH, 8888 for the optional JupyterLab UI
    support_public_ip=True,
    start_ssh=True,
    # NOTE: do NOT pass docker_args here. The runpod/pytorch image's default
    # entrypoint is what starts sshd (and JupyterLab). Overriding it with our
    # own command kills sshd before it ever starts.
)
POD_ID = pod['id']
print(f"Pod created: {POD_ID}")

# Wait for runtime info (public SSH IP+port)
SSH_HOST = SSH_PORT = None
for _ in range(60):
    info = runpod.get_pod(POD_ID)
    for p in (info.get('runtime') or {}).get('ports', []):
        if p.get('privatePort') == 22 and p.get('isIpPublic'):
            SSH_HOST, SSH_PORT = p['ip'], p['publicPort']
            break
    if SSH_HOST: break
    time.sleep(5)
    print(f"  ... waiting for SSH endpoint")

if not SSH_HOST:
    raise RuntimeError("Pod never exposed a public SSH endpoint. Check the RunPod console.")
print(f"SSH endpoint: root@{SSH_HOST}:{SSH_PORT}")

## 5. Wait for SSH to actually accept connections

Above, the *endpoint* exists. Now we wait until sshd inside the container is taking connections (~30–60 s).

In [ ]:
SSH_BASE = ['ssh', '-p', str(SSH_PORT),
            '-o', 'StrictHostKeyChecking=no',
            '-o', 'UserKnownHostsFile=/dev/null',
            '-o', 'LogLevel=ERROR',
            '-o', 'ConnectTimeout=5',
            f'root@{SSH_HOST}']
SCP_BASE = ['scp', '-P', str(SSH_PORT),
            '-o', 'StrictHostKeyChecking=no',
            '-o', 'UserKnownHostsFile=/dev/null',
            '-o', 'LogLevel=ERROR']

t0 = time.time()
for attempt in range(60):
    rc = subprocess.run(SSH_BASE + ['true'], stdout=subprocess.DEVNULL,
                        stderr=subprocess.DEVNULL).returncode
    if rc == 0:
        print(f"SSH up after {int(time.time()-t0)}s")
        break
    time.sleep(5)
    print(f"  ... waiting for sshd ({int(time.time()-t0)}s)")
else:
    raise RuntimeError("SSH never accepted connections. Check pod logs in the RunPod console.")

  ... waiting for sshd (5s)
  ... waiting for sshd (10s)
  ... waiting for sshd (15s)
  ... waiting for sshd (19s)
  ... waiting for sshd (25s)
  ... waiting for sshd (30s)
  ... waiting for sshd (35s)


## 6. Helper: stream output from remote commands

Two small wrappers over `subprocess`:
- `ssh_run(cmd)` — runs `cmd` on the pod, streams stdout to this notebook line by line, raises on nonzero exit.
- `scp_upload(local, remote)` — copies one file from your laptop to the pod.

In [ ]:
def ssh_run(cmd, stream=True):
    proc = subprocess.Popen(SSH_BASE + [cmd], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        if stream: print(line, end='', flush=True)
        out.append(line)
    proc.wait()
    if proc.returncode:
        raise RuntimeError(f"remote command failed (rc={proc.returncode}): {cmd[:100]}...")
    return ''.join(out)

def scp_upload(local, remote):
    subprocess.run(SCP_BASE + [str(local), f'root@{SSH_HOST}:{remote}'], check=True)
    print(f"  uploaded {pathlib.Path(local).name} -> {remote}")

## 7. Upload the SEC parquets to the pod

In [ ]:
ssh_run("mkdir -p /workspace/sec_data")
for fname in PARQUETS:
    src = DATA_DIR / fname
    if not src.exists():
        raise FileNotFoundError(src)
    print(f"Uploading {fname} ({src.stat().st_size/1e6:.1f} MB)...")
    scp_upload(src, f"/workspace/sec_data/{fname}")
print("done.")

## 8. Install nanochat on the pod

Clones karpathy/nanochat, installs deps with `uv` (fast, ~1 min on A100 box), and stages the SEC parquets into nanochat's expected `~/.cache/nanochat/base_data_climbmix/` layout (one shard per parquet plus a final val shard).

In [ ]:
INSTALL = '''
set -e
cd /workspace
test -d nanochat || git clone https://github.com/karpathy/nanochat.git
curl -LsSf https://astral.sh/uv/install.sh | sh >/dev/null 2>&1
export PATH="$HOME/.local/bin:$PATH"
cd nanochat
uv venv
uv sync --extra gpu
'''
ssh_run(INSTALL)

In [ ]:
# Stage SEC parquets into nanochat's data dir.
# nanochat reads ~/.cache/nanochat/base_data_climbmix/shard_NNNNN.parquet
# (last shard = val). We re-export each input parquet keeping only `text`,
# split into N+1 shards (last one held out for val).
PREP = '''
set -e
source /workspace/nanochat/.venv/bin/activate
python - <<'PY'
import pathlib, pandas as pd, pyarrow as pa, pyarrow.parquet as pq, os
src_dir = pathlib.Path("/workspace/sec_data")
target = pathlib.Path(os.path.expanduser("~/.cache/nanochat/base_data_climbmix"))
target.mkdir(parents=True, exist_ok=True)
for old in target.glob("shard_*.parquet"): old.unlink()

frames = []
for p in sorted(src_dir.glob("*.parquet")):
    df = pd.read_parquet(p, columns=["text"])
    df = df[df["text"].str.len() > 200].reset_index(drop=True)
    print(f"  {p.name}: {len(df):,} docs after filtering")
    frames.append(df)
big = pd.concat(frames, ignore_index=True).sample(frac=1.0, random_state=42).reset_index(drop=True)
print(f"Total docs: {len(big):,}")

NUM = 4
sz = (len(big) + NUM - 1) // NUM
for i in range(NUM):
    chunk = big.iloc[i*sz:(i+1)*sz]
    if len(chunk) == 0: continue
    out = target / f"shard_{i:05d}.parquet"
    pq.write_table(pa.Table.from_pandas(chunk[["text"]]), out)
    print(f"  wrote {out.name} rows={len(chunk):,} size={out.stat().st_size/1e6:.1f}MB")
PY
'''
ssh_run(PREP)

## 9. Train a custom BPE tokenizer (~30 s)

This trains a small BPE tokenizer (vocab 4096) on ~20 M chars of SEC text. Identical to nanochat's `scripts/tok_train.py`, just with smaller `--max-chars` and `--vocab-size`.

In [ ]:
ssh_run(f'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python -m scripts.tok_train --max-chars {TOK_MAX_CHARS} --vocab-size {VOCAB_SIZE}
''')

## 10. Pretrain the GPT (~5–7 min)

The actual training run. You'll see `step N/{NUM_STEPS} | loss=... | tok/s=...` lines streaming in. Loss should drop from ~7-8 down to ~3-4.

In [ ]:
ssh_run(f'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python -m scripts.base_train \
    --depth {DEPTH} \
    --max-seq-len {MAX_SEQ_LEN} \
    --device-batch-size {DEV_BATCH} \
    --total-batch-size {TOTAL_BATCH} \
    --num-iterations {NUM_STEPS} \
    --eval-tokens 4096 \
    --eval-every 100 \
    --sample-every 100 \
    --core-metric-every -1 \
    --run dummy
''')

## 11. Sample generations from your model

Loads the just-saved checkpoint and prints completions for a few SEC-flavored prompts.

⚠️ This is a **base** model (no instruction tuning). Treat each prompt as the *start* of a passage that the model continues — not a question.

In [ ]:
SAMPLE = r'''
set -e
source /workspace/nanochat/.venv/bin/activate
cd /workspace/nanochat
python - <<'PY'
import os
os.environ.setdefault("MASTER_ADDR","localhost"); os.environ.setdefault("MASTER_PORT","29500")
os.environ.setdefault("RANK","0"); os.environ.setdefault("WORLD_SIZE","1"); os.environ.setdefault("LOCAL_RANK","0")
from nanochat.checkpoint_manager import load_model
from nanochat.engine import Engine
from nanochat.common import autodetect_device_type, compute_init
dt = autodetect_device_type()
_,_,_,_,device = compute_init(dt)
model, tok, meta = load_model("base", device, phase="eval")
engine = Engine(model, tok)

PROMPTS = [
    "ITEM 1A. RISK FACTORS\n\nThe following risks could materially affect our",
    "Our business is focused on",
    "We may be unable to",
    "Management's Discussion and Analysis of Financial Condition\n\nOverview:",
]
for p in PROMPTS:
    print("\n" + "-"*70)
    print("Prompt:", repr(p)); print("Continuation:")
    print(p, end="")
    toks = [tok.get_bos_token_id()] + tok.encode(p)
    for tc, _ in engine.generate(toks, num_samples=1, max_tokens=120, temperature=0.8, top_k=50):
        print(tok.decode([tc[0]]), end="", flush=True)
    print()
PY
'''
ssh_run(SAMPLE)

## 12. Chat with your model

`chat(prompt)` sends `prompt` to the pod, generates a continuation, and returns it. Edit and re-run the cell with whatever prompt you like.

Each call reloads the model on the pod (~5 s overhead). Good enough for a few demo prompts; if you want a faster interactive loop, you'd start an inference server on the pod, but that's out of scope here.

In [ ]:
import shlex

def chat(prompt: str, max_tokens: int = 200, temperature: float = 0.8, top_k: int = 50) -> str:
    """Generate a continuation of `prompt` on the trained model."""
    py_prompt = prompt.replace("\\", "\\\\").replace("'", "\\'")
    code = f'''
import os
os.environ.setdefault("MASTER_ADDR","localhost"); os.environ.setdefault("MASTER_PORT","29500")
os.environ.setdefault("RANK","0"); os.environ.setdefault("WORLD_SIZE","1"); os.environ.setdefault("LOCAL_RANK","0")
from nanochat.checkpoint_manager import load_model
from nanochat.engine import Engine
from nanochat.common import autodetect_device_type, compute_init
dt = autodetect_device_type()
_,_,_,_,device = compute_init(dt)
model, tok, meta = load_model("base", device, phase="eval")
engine = Engine(model, tok)
prompt = {prompt!r}
toks = [tok.get_bos_token_id()] + tok.encode(prompt)
for tc, _ in engine.generate(toks, num_samples=1, max_tokens={max_tokens}, temperature={temperature}, top_k={top_k}):
    print(tok.decode([tc[0]]), end="", flush=True)
print()
'''
    full = (
        "source /workspace/nanochat/.venv/bin/activate && "
        "cd /workspace/nanochat && "
        f"python -c {shlex.quote(code)}"
    )
    print(prompt, end="")
    out = ssh_run(full)
    return out

_ = chat("ITEM 1A. RISK FACTORS\n\nThe primary risks include")

In [ ]:
# Try your own prompt — edit and re-run.
_ = chat("Our principal sources of revenue are")

## 13. Terminate the pod

⚠️ **Don't skip this.** Idle GPU pods cost real money.

In [ ]:
runpod.terminate_pod(POD_ID)
print(f"Terminated pod {POD_ID}.")